# Gaussian Mixture Modelling - Error Deconvolution

In this notebook I shall demonstrate how to use the machinery defined in this rust crate 

In [ ]:
import essrs
import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
import time
from typing import List

In [ ]:
N_SAMPLES: int = 250
N_BURN_IN: int = 200

In [ ]:
N: List[int] = [2000, 4000]

# Mean and covariance
MU = [
    np.array([-2.3, 2.7]),
    np.array([3.0, 1.5])
]

X_SIGMA: float = 2.0
Y_SIGMA: float = 3.5
CORR: float = 0.55

Sigma = np.array([
    [X_SIGMA * X_SIGMA, X_SIGMA * Y_SIGMA * CORR],
    [X_SIGMA * Y_SIGMA * CORR, Y_SIGMA * Y_SIGMA]
])

# Cholesky decomposition: Sigma = L @ L.T
L = np.linalg.cholesky(Sigma)

In [ ]:
DATA_X = np.empty(shape=(0,2))

for n, mu in zip(N, MU):

    # Sample N independent standard 2D Gaussians
    Z = np.random.randn(n, 2)

    # Transform to N(mu, Sigma)
    DATA_X = np.concatenate((DATA_X, mu + Z @ L.T), axis=0)


In [ ]:
plt.figure()
plt.scatter(DATA_X[:, 0], DATA_X[:, 1], s=0.5, marker=".")
plt.show()

In [ ]:
mu_x = np.mean(DATA_X[:, 0])
mu_y = np.mean(DATA_X[:, 1])
cov_mat = np.cov(DATA_X.T)
print(cov_mat.shape)
print(cov_mat)

x_std = np.sqrt(cov_mat[0, 0])
y_std = np.sqrt(cov_mat[1, 1])

print(x_std, y_std)

p = cov_mat[0, 1] / (x_std * y_std)
print(p)

N_PARAMETERS: int = 11
N_SUB_PARAMETERS: int = 5
N_WALKERS: int = 50

A = np.array([mu_x, mu_y, x_std, y_std, p]) * np.random.normal(1, 0.05, size=(N_WALKERS, N_SUB_PARAMETERS))
B = np.array([mu_x, mu_y, x_std, y_std, p]) * np.random.normal(1, 0.05, size=(N_WALKERS, N_SUB_PARAMETERS))

A[:, 0] -= 1
B[:, 0] += 1
ics = np.column_stack((np.random.normal(0.5, 0.05, N_WALKERS), A, B))

print(ics)

In [ ]:
model = essrs.model.scientific.normal_2d_gmm_deconv(n_components=2, data=np.column_stack((DATA_X, np.zeros((DATA_X.shape[0], 3)))))

burn_move = essrs.moves.hybrid_ddc_move(one_in_n=10)
sampler_move = essrs.moves.differential_move()
config = essrs.EnsembleConfig(n_samples=N_SAMPLES, n_walkers=N_WALKERS, n_parameters=N_PARAMETERS, n_burn_in=N_BURN_IN)

start = time.perf_counter()
fit, ll = essrs.run_mcmc(
    log_likelihood=model,
    config=config,
    burn_in_move=burn_move,
    sampling_move=sampler_move,
    initial_conditions=ics
)

elapsed = time.perf_counter() - start
print(f"ESSRS took {elapsed:.3f} seconds")

In [ ]:
print("Result:")
for i, label in enumerate(["mean X", "mean Y", "sigma X", "sigma Y", "p_corr"]):
    print(f"{label} => {np.mean(fit[-1, :, i].ravel()):7.5f} +- {np.std(fit[-1, :, i].ravel()):7.5f}")


In [ ]:
x = np.linspace(-15, 15, 300)
y = np.linspace(-15, 15, 300)

dxdy = (x[1] - x[0]) * (y[1] - y[0])

X, Y = np.meshgrid(x, y)

mu = [0, 0]
cov = [[1, 0.5],
       [0.5, 2]]

Z = multivariate_normal.pdf(
    np.stack([X, Y], axis=-1),
    mean=mu,
    cov=cov
)

plt.figure()
plt.scatter(DATA_X[:, 0], DATA_X[:, 1], s=0.1, marker=".", color="r", alpha=0.2)

for theta in fit[-1, :, :]:
    Z = theta[0] * multivariate_normal.pdf(
        np.stack([X, Y], axis=-1),
        mean=[theta[1], theta[2]],
        cov=[
            [theta[3]**2, theta[3]*theta[4]*theta[5]],
            [theta[3]*theta[4]*theta[5], theta[4]**2]
        ]
    )

    Z += (1.0 - theta[0]) * multivariate_normal.pdf(
        np.stack([X, Y], axis=-1),
        mean=[theta[6], theta[7]],
        cov=[
            [theta[8]**2, theta[8]*theta[9]*theta[10]],
            [theta[8]*theta[9]*theta[10], theta[9]**2]
        ]
    )

    Z /= np.sum(Z) * dxdy

    plt.contour(X, Y, Z, alpha=0.3, cmap="Blues")


plt.xlim(-7, 10)
plt.ylim(-9,11)
plt.show()